## Initialisation of rank metric calculation

In [1]:
# set the basepath to "one up"
basepath="../"
## set the path
import sys
sys.path.append(basepath+"ppi.lib/")
import ml_lib as mlib
## We use mapper.extract4ml to load the data.
## the default mode uses the transdoremd data as inpuit matrix X.
## The function allows us to specify several column selectors to assess the 
## information content in different feature sets.
## 1) column names for different processing steps
## mlib.col4tmlab="Transmembrane"                      ## column name in classic_raw holding the transmembran labels                                          
## mlib.col4eratelab="GMM_Label_Combined"              ## column name in classic_raw holding the evolutionary rate labels  
##
## input label columns to be used in function extract4ml as parameters Xcols with discardXcols=True
## mlib.rem_mns_fnam="../source/colnams_remove_means.txt"      ## column names containing mean values
## mlib.rem_sums_fnam="../source/colnams_remove_sums.txt"      ## column names containing sum values
## input label columns to be used in function extract4ml as parameters Xcols with discardXcols=True
## mlib.keep_but_small_counts="../source/colnamesdropsmallcounts.txt" ## retains all but small counts 
## mlib.keep_mean_props="../source/collnamesselmeanprops.txt"         ## retains mean values and proportions
## mlib.keep_sum_props="../source/colnamesselsumprops.txt"            ## retains sum values and proportions (including all counts)
## to load a desired column set use function mlib.getcolnams() and provide the filename as argument.
## The columm names should be provided to function mlib.extract4ml together with an appropriately set flag discardXcols which
## controls whether the columns should be taken or removed.
## start with all data:
(X, y, ftrnams, sampleids)=mlib.extract4ml()

### Normalization with standard scaler

In [2]:
import numpy as np
from sklearn.preprocessing import StandardScaler as SSC
## we transform the data as a whole
Xtr=SSC().fit_transform(X)
## convert the labels
from sklearn.preprocessing import LabelEncoder as LBE
ytr=LBE().fit_transform(y)
## we work now with Xtr and ytr

## Initialise generic ML tools 

In [ ]:
## set the path
import sys
sys.path.append(basepath+"ppi.lib/")
import ml_lib as mlib
import importlib
importlib.reload(mlib)
## import the sklearn model selection and assessment tools
from sklearn.model_selection import cross_val_predict as cvp
from sklearn.model_selection import StratifiedKFold as SKF
from sklearn.model_selection import GridSearchCV as GSCV

nfold=10
## create two SKF instances for grid search cv and cvp
skfo=SKF(nfold, shuffle=True)
skfi=SKF(nfold, shuffle=True)

## use skfo and cvp to create defaut predictions.
y_def=cvp(mlib.DefClassifier(), Xtr, ytr, cv=skfo)
acc_def=100.0*np.sum(y_def==ytr)/ytr.shape[0]

## Evaluation of Feature relevance
This evasluation uses six diffrerent approaches
* Average feature importances of a random forest classifier (fitted_rfc.feature_importances_)
* Feature sepcific Bayes factors of a GPC when comparing individual feature values concatenated with "1" against "1" based inputs
* Feature specific McNemar p-values of a SVC based predictions against default predictions (majority vote)
* Feature specific modified Shannon channel capacities of SVC based probabilities
* Feature specific AUC values (derived from SVC)
* Feature specific generalisation accuracies (derived from SVC)

In [8]:
## random forest approach
print("RFC")
from sklearn.ensemble import RandomForestClassifier as RFC
rfc_grid={"n_estimators":[100, 200],
          "criterion":["gini"],
          "max_depth":[5, 7, 9]
         }
rfcinit=RFC()
## define the gpc grid search
gsrfc=GSCV(rfcinit, rfc_grid, n_jobs=-1, cv=skfi)
gsrfc.fit(Xtr, ytr)
rfc=gsrfc.best_estimator_
feature_eval={"names":ftrnams, "RFC_gini_importance":rfc.feature_importances_.tolist()}

rfc_grid={"n_estimators":[100, 200],
          "criterion":["entropy"],
          "max_depth":[5, 7, 9]
         }
rfcinit=RFC()
## define the gpc grid search
gsrfc=GSCV(rfcinit, rfc_grid, n_jobs=-1, cv=skfi)
gsrfc.fit(Xtr, ytr)
rfc=gsrfc.best_estimator_
feature_eval["RFC_entropy_importance"]=rfc.feature_importances_.tolist()

rfc_grid={"n_estimators":[100, 200],
          "criterion":["log_loss"],
          "max_depth":[5, 7, 9]
         }
rfcinit=RFC()
## define the gpc grid search
gsrfc=GSCV(rfcinit, rfc_grid, n_jobs=-1, cv=skfi)
gsrfc.fit(Xtr, ytr)
rfc=gsrfc.best_estimator_
feature_eval["RFC_logloss_importance"]=rfc.feature_importances_.tolist()


RFC


In [9]:
## GPC based importances
print("GPC")
from sklearn.gaussian_process import GaussianProcessClassifier as GPC
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ConstantKernel
minval=10.0**-150
maxval=10.0**150
in_ones=np.ones((Xtr.shape[0], 1))
gpcmdl=GPC(kernel=RBF(length_scale_bounds=(minval, maxval)))
gpcmdl.fit(in_ones, ytr)
null_lmlh=gpcmdl.log_marginal_likelihood_value_
alt_lmlhs=[]
for cdx in range(Xtr.shape[1]):
    print("it {0} of {1}".format(cdx, Xtr.shape[1]))
    gpcmdl=GPC(kernel=RBF(length_scale_bounds=(minval, maxval)))
    Xcr=np.concatenate((in_ones, Xtr[:,[cdx]]), axis=1)
    gpcmdl.fit(Xcr, ytr)
    alt_lmlhs.append(gpcmdl.log_marginal_likelihood_value_)

bayes_fctrs=np.array(alt_lmlhs)-null_lmlh
feature_eval["Bayes_logodds"]=bayes_fctrs.tolist()

GPC
it 0 of 64
it 1 of 64
it 2 of 64
it 3 of 64
it 4 of 64
it 5 of 64
it 6 of 64
it 7 of 64
it 8 of 64
it 9 of 64
it 10 of 64
it 11 of 64
it 12 of 64
it 13 of 64
it 14 of 64
it 15 of 64
it 16 of 64
it 17 of 64
it 18 of 64
it 19 of 64
it 20 of 64
it 21 of 64
it 22 of 64
it 23 of 64
it 24 of 64
it 25 of 64
it 26 of 64
it 27 of 64
it 28 of 64
it 29 of 64
it 30 of 64
it 31 of 64
it 32 of 64
it 33 of 64
it 34 of 64
it 35 of 64
it 36 of 64
it 37 of 64
it 38 of 64
it 39 of 64
it 40 of 64
it 41 of 64
it 42 of 64
it 43 of 64
it 44 of 64
it 45 of 64
it 46 of 64
it 47 of 64
it 48 of 64
it 49 of 64
it 50 of 64
it 51 of 64
it 52 of 64
it 53 of 64
it 54 of 64
it 55 of 64
it 56 of 64
it 57 of 64
it 58 of 64
it 59 of 64
it 60 of 64
it 61 of 64
it 62 of 64
it 63 of 64


In [ ]:
## SVC based evaluations
print("SVC")
from sklearn.svm import SVC

## use skfo and cvp to create defaut predictions.
y_def=cvp(mlib.DefClassifier(), Xtr, ytr, cv=skfo)
acc_def=100.0*np.sum(y_def==ytr)/ytr.shape[0]

## hyperparameters of SVC: C=1.0, gamma="auto", "scale" and float values
minC=10**-6
maxC=50
Cvals=7
mingam=10**-2
maxgam=10
gamvals=5
## hyperparameters for rbf kernel based SVC
svcgrid={"C":np.linspace(minC, maxC, Cvals),
         "gamma":["auto", "scale"]+np.linspace(mingam, maxgam, gamvals).tolist()}

svc_acc=[]
svc_auc=[]
svc_shannon=[]
svc_mcnemar=[]

for cdx in range(Xtr.shape[1]):
    print("it {0} of {1}".format(cdx, Xtr.shape[1]))
    ## define the SVC with rbf lernes alnd such that probabilities get estimated.
    gssvc=GSCV(SVC(probability=True), svcgrid, n_jobs=-1, cv=skfi)
    Xcr=np.concatenate((in_ones, Xtr[:,[cdx]]), axis=1)
    ## unbiased predictions by nested cross validation using mlib.crossvalprobs
    P_svc=mlib.crossvalprobs(gssvc, Xcr, ytr, cv=skfo, verbosity=False)
    rocx_svc, rocy_svc, auc_svc, shannon_svc, acc_svc, pval_svc=mlib.roc_auc_shannon_acc_mcnemar(P_svc[:,1], ytr)
    ## collect all feature specific metrics
    svc_acc.append(acc_svc)
    svc_auc.append(auc_svc)
    svc_shannon.append(shannon_svc)
    svc_mcnemar.append(pval_svc)
## add results to feature assessment
feature_eval["SVC_acc"]=svc_acc
feature_eval["SVC_auc"]=svc_auc
feature_eval["SVC_Shannon"]=svc_shannon
feature_eval["SVC_McNemar"]=svc_mcnemar

import pandas as pd
dfrm=pd.DataFrame(feature_eval)
dfrm.to_csv("../results/physicoftrs4transstate_eval.csv", index=False)
print("done")

SVC
it 0 of 64
it 1 of 64
it 2 of 64
it 3 of 64
it 4 of 64
it 5 of 64
it 6 of 64
it 7 of 64
it 8 of 64
it 9 of 64
it 10 of 64
it 11 of 64
it 12 of 64
it 13 of 64
it 14 of 64
it 15 of 64
it 16 of 64
it 17 of 64
it 18 of 64
it 19 of 64
it 20 of 64
it 21 of 64
it 22 of 64
it 23 of 64
it 24 of 64
it 25 of 64
it 26 of 64
it 27 of 64
it 28 of 64
it 29 of 64
it 30 of 64
it 31 of 64
it 32 of 64
it 33 of 64
it 34 of 64
it 35 of 64
it 36 of 64
it 37 of 64
it 38 of 64
it 39 of 64
it 40 of 64
it 41 of 64
it 42 of 64
it 43 of 64
it 44 of 64
it 45 of 64
it 46 of 64
it 47 of 64
